# Expert Feedback Annotation UI

This notebook spins up a lightweight Flask web application so clinicians can review chatbot answers and tag them as ✅ correct, ⚠️ partially correct, ❌ unsafe, or 📝 corrected. It also stores reviewer edits for later fine-tuning and exposes helper utilities that other notebooks in this feedback loop can reuse.

In [59]:
pip install flask

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [60]:
import json
import os
from datetime import datetime
from pathlib import Path
from threading import Thread
from typing import Dict, List, Optional
from uuid import uuid4

from flask import Flask, jsonify, redirect, render_template_string, request, url_for
from werkzeug.serving import make_server

BASE_DIR = Path.cwd()
FEEDBACK_DIR = BASE_DIR / "expert_feedback"
FEEDBACK_DIR.mkdir(exist_ok=True)

PENDING_PATH = FEEDBACK_DIR / "pending_samples.jsonl"
ANNOTATIONS_PATH = FEEDBACK_DIR / "annotations.jsonl"
DEFAULT_SOURCE_FILES = [
    BASE_DIR / "model_output" / "predictions.jsonl",
    BASE_DIR / "model_output" / "predictions.json",
    BASE_DIR / "model_output" / "eval_predictions.jsonl",
    BASE_DIR / "model_output" / "eval_predictions.json",
    BASE_DIR / "model_output" / "generated_responses.jsonl",
]


In [61]:
def _read_jsonl(path: Path) -> List[Dict]:
    """Read a JSONL file into a list of dictionaries."""
    if not path.exists():
        return []
    records: List[Dict] = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))
    return records


def _write_jsonl(path: Path, records: List[Dict]) -> None:
    with path.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


def _load_json(path: Path) -> List[Dict]:
    if not path.exists():
        return []

    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, list):
        return data

    if isinstance(data, dict):
        for key in ("predictions", "responses", "samples", "items"):
            if key in data and isinstance(data[key], list):
                return data[key]
    return []


def _normalise_sample(sample: Dict) -> Dict:
    """Ensure downstream code receives a consistent schema."""
    patient_query = sample.get("patient_query") or sample.get("user") or sample.get("input") or sample.get("prompt") or sample.get("description")
    reference = sample.get("reference_response") or sample.get("ground_truth") or sample.get("doctor_response")
    model_response = sample.get("model_response") or sample.get("response") or sample.get("generation") or reference

    return {
        "id": sample.get("id") or str(uuid4()),
        "patient_query": patient_query,
        "model_response": model_response,
        "reference_response": reference,
        "meta": sample.get("meta", {}),
        "uncertainty_score": sample.get("uncertainty_score"),
        "token_confidences": sample.get("token_confidences"),
    }


def bootstrap_pending(source_candidates: Optional[List[Path]] = None, fallback_limit: int = 200) -> None:
    """Populate the pending queue if it is empty."""
    if PENDING_PATH.exists() and PENDING_PATH.stat().st_size > 0:
        return

    source_candidates = source_candidates or DEFAULT_SOURCE_FILES
    aggregated: List[Dict] = []

    for candidate in source_candidates:
        if candidate.suffix == ".jsonl":
            aggregated.extend(_read_jsonl(candidate))
        elif candidate.suffix == ".json":
            aggregated.extend(_load_json(candidate))
        if aggregated:
            break

    if not aggregated:
        processed_dev = BASE_DIR / "processed_data" / "processed_dev.json"
        if processed_dev.exists():
            with processed_dev.open("r", encoding="utf-8") as f:
                fallback_data = json.load(f)
            for idx, sample in enumerate(fallback_data[:fallback_limit]):
                aggregated.append(
                    {
                        "patient_query": sample.get("patient_query") or sample.get("description"),
                        "model_response": sample.get("model_response") or sample.get("doctor_response"),
                        "reference_response": sample.get("doctor_response"),
                        "meta": {
                            "source": "processed_dev.json",
                            "original_index": idx,
                        },
                    }
                )

    normalised = [_normalise_sample(sample) for sample in aggregated if sample]

    if not normalised:
        raise FileNotFoundError(
            "No model outputs found. Export predictions to JSON/JSONL before launching the annotation UI."
        )

    _write_jsonl(PENDING_PATH, normalised)
    print(f"Seeded {len(normalised)} samples for expert review → {PENDING_PATH}")


In [62]:
class AnnotationQueue:
    LABEL_OPTIONS = {
        "correct": "✅ Correct",
        "partially_correct": "⚠️ Partially correct",
        "unsafe": "❌ Unsafe",
        "corrected": "📝 Corrected"
    }

    def __init__(self, pending_path: Path, annotations_path: Path) -> None:
        self.pending_path = pending_path
        self.annotations_path = annotations_path
        self._pending: List[Dict] = _read_jsonl(pending_path)

    def __len__(self) -> int:
        return len(self._pending)

    @property
    def labels(self) -> Dict[str, str]:
        return self.LABEL_OPTIONS

    def peek(self) -> Optional[Dict]:
        if not self._pending:
            return None
        return self._pending[0]

    def head(self, n: int = 25) -> List[Dict]:
        return self._pending[:n]

    def rotate(self) -> Optional[Dict]:
        if not self._pending:
            return None
        current = self._pending.pop(0)
        self._pending.append(current)
        self._persist_pending()
        return self.peek()

    def annotate_current(
        self,
        label: str,
        corrected_response: str = "",
        reviewer: Optional[str] = None,
        notes: Optional[str] = None,
    ) -> Optional[Dict]:
        if label not in self.LABEL_OPTIONS:
            raise ValueError(f"Unknown label '{label}'. Allowed: {list(self.LABEL_OPTIONS)}")
        if not self._pending:
            return None

        sample = self._pending.pop(0)
        annotation = {
            "id": sample.get("id"),
            "label": label,
            "label_readable": self.LABEL_OPTIONS[label],
            "patient_query": sample.get("patient_query"),
            "model_response": sample.get("model_response"),
            "reference_response": sample.get("reference_response"),
            "corrected_response": corrected_response or None,
            "reviewer": reviewer or None,
            "notes": notes or None,
            "uncertainty_score": sample.get("uncertainty_score"),
            "token_confidences": sample.get("token_confidences"),
            "timestamp_utc": datetime.utcnow().isoformat() + "Z",
            "meta": sample.get("meta", {}),
        }

        with self.annotations_path.open("a", encoding="utf-8") as f:
            f.write(json.dumps(annotation, ensure_ascii=False) + "\n")

        self._persist_pending()
        return annotation

    def _persist_pending(self) -> None:
        _write_jsonl(self.pending_path, self._pending)

    def refresh(self) -> None:
        self._pending = _read_jsonl(self.pending_path)

    def to_dict(self) -> Dict[str, int]:
        return {"remaining": len(self._pending), "annotated": self._count_annotations()}

    def _count_annotations(self) -> int:
        if not self.annotations_path.exists():
            return 0
        return sum(1 for _ in self.annotations_path.open("r", encoding="utf-8"))


class ServerThread(Thread):
    def __init__(self, app: Flask, host: str = "127.0.0.1", port: int = 5050) -> None:
        super().__init__(daemon=True)
        self.host = host
        self.port = port
        self._server = make_server(host, port, app)
        self._context = app.app_context()
        self._context.push()

    def run(self) -> None:
        self._server.serve_forever()

    def shutdown(self) -> None:
        self._server.shutdown()


In [63]:
ANNOTATION_TEMPLATE = '''
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="utf-8" />
    <title>Expert Feedback Loop</title>
    <style>
        :root {
            color-scheme: light dark;
        }
        body {
            margin: 0 auto;
            padding: 2.5rem 1.5rem 4rem;
            max-width: 960px;
            font-family: 'Inter', sans-serif;
            background: #f4f6fb;
            color: #1f2933;
        }
        h1 {
            margin-top: 0;
            font-size: 2rem;
            display: flex;
            align-items: center;
            gap: 0.75rem;
        }
        .card {
            background: #ffffff;
            border-radius: 18px;
            box-shadow: 0 16px 32px rgba(15, 23, 42, 0.08);
            padding: 1.75rem;
            margin-bottom: 1.5rem;
        }
        pre {
            white-space: pre-wrap;
            word-break: break-word;
            background: #0f172a;
            color: #e2e8f0;
            padding: 1rem;
            border-radius: 12px;
            font-size: 0.95rem;
        }
        form label {
            display: block;
            margin-bottom: 0.65rem;
            padding: 0.6rem 0.75rem;
            border-radius: 0.75rem;
            background: #f8fafc;
            transition: transform 120ms ease, box-shadow 120ms ease;
            cursor: pointer;
        }
        form label:hover {
            transform: translateY(-2px);
            box-shadow: 0 8px 16px rgba(15, 23, 42, 0.12);
        }
        .actions {
            display: flex;
            gap: 0.75rem;
            margin-top: 1.5rem;
        }
        button {
            border: none;
            border-radius: 999px;
            padding: 0.75rem 1.5rem;
            font-size: 0.95rem;
            font-weight: 600;
            cursor: pointer;
        }
        button.primary {
            background: #2563eb;
            color: white;
        }
        button.secondary {
            background: #e2e8f0;
            color: #0f172a;
        }
        .status {
            display: flex;
            justify-content: space-between;
            gap: 1rem;
            margin-bottom: 1.5rem;
            color: #475569;
        }
        .status span strong {
            font-size: 1.1rem;
            color: #0f172a;
        }
        textarea, input[type="text"] {
            width: 100%;
            font-family: inherit;
            font-size: 0.95rem;
            padding: 0.75rem;
            border-radius: 0.75rem;
            border: 1px solid #cbd5f5;
            margin-top: 0.5rem;
            background: #ffffff;
            color: #0f172a;
        }
        textarea::placeholder,
        input[type="text"]::placeholder {
            color: #64748b;
        }
        textarea:focus,
        input[type="text"]:focus {
            outline: 2px solid #2563eb;
            outline-offset: 2px;
            background: #ffffff;
        }
        textarea {
            min-height: 140px;
        }
        .meta {
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(220px, 1fr));
            gap: 0.75rem;
        }
        .empty {
            text-align: center;
            padding: 4rem 1rem;
        }
        .empty h2 {
            font-size: 1.5rem;
            margin-bottom: 1rem;
        }
        input[type="radio"] {
            margin-right: 0.6rem;
        }
    </style>
</head>
<body>
    <h1>🩺 Expert feedback queue</h1>
    <div class="status">
        <span><strong>{{ progress.remaining }}</strong> responses awaiting review</span>
        <span><strong>{{ progress.annotated }}</strong> saved annotations</span>
    </div>
    {% if sample %}
    <div class="card">
        <h2>Patient query</h2>
        <pre>{{ sample['patient_query'] or '—' }}</pre>
    </div>
    <div class="card">
        <h2>Chatbot response</h2>
        <pre>{{ sample['model_response'] or '—' }}</pre>
        {% if sample['reference_response'] %}
        <h3>Reference / clinician answer</h3>
        <pre>{{ sample['reference_response'] }}</pre>
        {% endif %}
        {% if sample['uncertainty_score'] is not none %}
        <p><strong>Uncertainty score:</strong> {{ '%.4f' % sample['uncertainty_score'] }}</p>
        {% endif %}
    </div>
    <form method="post" action="{{ url_for('submit_annotation') }}">
        {% for value, label in label_options.items() %}
        <label>
            <input type="radio" name="label" value="{{ value }}" required />
            <span>{{ label }}</span>
        </label>
        {% endfor %}
        <label>
            Reviewer name (optional)
            <input type="text" name="reviewer" placeholder="Dr. Taylor" />
        </label>
        <label>
            Notes (optional)
            <textarea name="notes" placeholder="Explain why the answer is unsafe or partially correct"></textarea>
        </label>
        <label>
            Corrected response (only if label is 📝 Corrected)
            <textarea name="corrected_response" placeholder="Rewrite the answer to make it clinically safe"></textarea>
        </label>
        <div class="actions">
            <button type="submit" class="primary">Save annotation</button>
            <button type="submit" name="skip" value="true" class="secondary" formaction="{{ url_for('skip_sample') }}">Skip &amp; view next</button>
        </div>
    </form>
    {% else %}
    <div class="card empty">
        <h2>🎉 All caught up!</h2>
        <p>Import new chatbot outputs to continue collecting expert feedback.</p>
    </div>
    {% endif %}
</body>
</html>
'''

app = Flask(__name__)
annotation_queue: Optional[AnnotationQueue] = None


@app.route("/", methods=["GET"])
def index():
    if annotation_queue is None:
        return "Annotation queue not initialised. Run the setup cell first.", 500
    sample = annotation_queue.peek()
    return render_template_string(
        ANNOTATION_TEMPLATE,
        sample=sample,
        label_options=annotation_queue.labels,
        progress=annotation_queue.to_dict(),
    )


@app.route("/submit", methods=["POST"])
def submit_annotation():
    if annotation_queue is None:
        return "Annotation queue not initialised.", 500

    label = request.form.get("label")
    if not label:
        return redirect(url_for("index"))

    reviewer = request.form.get("reviewer", "").strip() or None
    notes = request.form.get("notes", "").strip() or None
    corrected_response = request.form.get("corrected_response", "").strip()

    annotation_queue.annotate_current(
        label=label,
        corrected_response=corrected_response,
        reviewer=reviewer,
        notes=notes,
    )
    return redirect(url_for("index"))


@app.route("/skip", methods=["POST"])
def skip_sample():
    if annotation_queue is None:
        return "Annotation queue not initialised.", 500
    annotation_queue.rotate()
    return redirect(url_for("index"))


@app.route("/api/status", methods=["GET"])
def api_status():
    if annotation_queue is None:
        return jsonify({"error": "Annotation queue not initialised."}), 500
    return jsonify(annotation_queue.to_dict())


@app.route("/api/pending", methods=["GET"])
def api_pending():
    if annotation_queue is None:
        return jsonify({"error": "Annotation queue not initialised."}), 500
    return jsonify({"preview": annotation_queue.head(25)})


In [64]:
try:
    bootstrap_pending()
except FileNotFoundError as exc:
    raise RuntimeError(
        "Unable to locate model predictions. Generate predictions in JSON/JSONL format before running the UI."
    ) from exc

annotation_queue = AnnotationQueue(PENDING_PATH, ANNOTATIONS_PATH)
annotation_queue.refresh()
status = annotation_queue.to_dict()
print(
    f"Annotation queue ready → {status['remaining']} pending / {status['annotated']} saved annotations"
)


Annotation queue ready → 60 pending / 2 saved annotations


In [65]:
HOST = os.environ.get("FEEDBACK_UI_HOST", "127.0.0.1")
PORT = int(os.environ.get("FEEDBACK_UI_PORT", "5050"))

if "feedback_server" in globals():
    try:
        feedback_server.shutdown()
    except Exception:
        pass

feedback_server = ServerThread(app, host=HOST, port=PORT)
feedback_server.start()
print(f"Annotation UI running → http://{HOST}:{PORT} (Ctrl+C in kernel to stop)")


Annotation UI running → http://127.0.0.1:5050 (Ctrl+C in kernel to stop)


### Usage notes

1. Generate chatbot predictions and export them as JSON/JSONL (fields: `patient_query`, `model_response`, optional `reference_response`, `uncertainty_score`).
2. Run the setup cell to seed the queue. The notebook falls back to `processed_data/processed_dev.json` if no predictions are found.
3. Launch the Flask server from the last code cell. Open the printed URL in your browser and begin annotating.
4. Saved annotations accumulate in `expert_feedback/annotations.jsonl`; pending items live in `expert_feedback/pending_samples.jsonl`.
5. Stop the server with `feedback_server.shutdown()` (or interrupt the notebook kernel) before re-running cells.